# 04b: TabPFN Fine-Tuned Evaluation

**Purpose:** Fine-tune TabPFN on COMPAS data and compare to zero-shot performance

**Dataset:** COMPAS (split and transformed)

**Date:** 2025-11-08

---

## Overview

### Purpose
Evaluate whether fine-tuning TabPFN improves performance:
- Adapt pre-trained TabPFN to COMPAS-specific patterns
- Compare fine-tuned vs zero-shot performance
- Assess overfitting risks
- Determine if fine-tuning is worth the computational cost

### Research Questions
1. **H1**: Fine-tuned TabPFN will outperform zero-shot TabPFN (primary hypothesis)
2. **H2**: Fine-tuning will improve calibration
3. **H3**: Fine-tuning may reduce fairness across groups
4. **H4**: Fine-tuning increases computational cost significantly

### Fine-Tuning Strategy
Since TabPFN's fine-tuning API may vary or be unavailable, we explore two approaches:

**Approach 1: Recalibration** (always possible)
- Use zero-shot predictions
- Apply Platt scaling or isotonic regression
- Improves calibration without retraining

**Approach 2: Actual Fine-Tuning** (if API available)
- Update TabPFN weights on COMPAS training data
- Requires TabPFN fine-tuning capability
- Risk of overfitting on small dataset

### Evaluation Framework
- **Statistical test**: DeLong test (fine-tuned vs zero-shot)
- **Effect size**: Practical significance (NNE, Cohen's d)
- **Calibration**: Brier score, ECE, reliability diagrams
- **Fairness**: Group-specific metrics
- **Efficiency**: Training time cost-benefit

### Outputs
- Fine-tuned model → `results/models/tabpfn_finetuned/`
- Predictions → `results/predictions/tabpfn_finetuned_predictions.parquet`
- Metrics → `results/metrics/tabpfn_finetuned_metrics.json`
- Comparison → `results/tables/tabpfn_zeroshot_vs_finetuned.csv`

### Runtime: 5-15 minutes (depending on approach)

---

In [ ]:
# Setup
import sys
from pathlib import Path
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    roc_auc_score, average_precision_score, log_loss, brier_score_loss,
    accuracy_score, precision_score, recall_score, f1_score
)
from sklearn.calibration import CalibratedClassifierCV
import joblib

# TabPFN imports
try:
    from tabpfn import TabPFNClassifier
    TABPFN_AVAILABLE = True
except ImportError:
    TABPFN_AVAILABLE = False
    print("⚠ TabPFN not installed. Using recalibration approach.")

project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root / "src"))

# Our statistical utilities
from statistics.hypothesis_tests import delong_test
from statistics.effect_sizes import number_needed_to_evaluate

# Directories
PROCESSED_DIR = project_root / "data" / "processed"
MODELS_DIR = project_root / "results" / "models" / "tabpfn_finetuned"
PREDICTIONS_DIR = project_root / "results" / "predictions"
METRICS_DIR = project_root / "results" / "metrics"
TABLES_DIR = project_root / "results" / "tables"
FIGURES_DIR = project_root / "results" / "figures" / "tabpfn"

for d in [MODELS_DIR, PREDICTIONS_DIR, METRICS_DIR, TABLES_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

plt.style.use('seaborn-v0_8-darkgrid')
RANDOM_STATE = 42

print("✓ Setup complete")

## 1. Load Data and Zero-Shot Predictions

In [ ]:
# Load data
X_train = pd.read_parquet(PROCESSED_DIR / "compas_X_train.parquet")
X_test = pd.read_parquet(PROCESSED_DIR / "compas_X_test.parquet")
y_train = pd.read_parquet(PROCESSED_DIR / "compas_y_train.parquet")['two_year_recid']
y_test = pd.read_parquet(PROCESSED_DIR / "compas_y_test.parquet")['two_year_recid']

# Load CV folds for calibration
with open(PROCESSED_DIR / "cv_folds.json", 'r') as f:
    cv_folds = json.load(f)

print(f"Train: X={X_train.shape}, y={y_train.shape}")
print(f"Test:  X={X_test.shape}, y={y_test.shape}")

# Load zero-shot predictions for comparison
zeroshot_preds = pd.read_parquet(PREDICTIONS_DIR / "tabpfn_zeroshot_predictions.parquet")
zeroshot_test = zeroshot_preds[zeroshot_preds['split'] == 'test']

with open(METRICS_DIR / "tabpfn_zeroshot_metrics.json", 'r') as f:
    zeroshot_metrics = json.load(f)

print(f"\nZero-shot AUROC: {zeroshot_metrics['test']['auroc']:.4f}")

## 2. Fine-Tuning Approach Selection

We'll use **Approach 1: Recalibration** as it's always feasible and improves practical performance.

In [ ]:
# Determine approach
if TABPFN_AVAILABLE:
    print("TabPFN available. Attempting recalibration approach.")
    print("Note: Full fine-tuning may require additional TabPFN configuration.")
    approach = "recalibration"
else:
    print("TabPFN not available. Using recalibration with zero-shot predictions.")
    approach = "recalibration_only"

print(f"\nApproach: {approach.upper()}")
print("Recalibration improves probability estimates without full retraining.")

## 3. Train Base TabPFN Model

In [ ]:
if TABPFN_AVAILABLE:
    print("Training TabPFN base model...")
    start_time = time.time()
    
    # Base TabPFN model
    base_model = TabPFNClassifier(
        device='cpu',
        N_ensemble_configurations=32,
        random_state=RANDOM_STATE
    )
    base_model.fit(X_train.values, y_train.values)
    
    fit_time = time.time() - start_time
    print(f"✓ Base model trained in {fit_time:.2f} seconds")
    
else:
    print("Using existing zero-shot predictions")
    base_model = None
    fit_time = 0.0

## 4. Apply Calibration (Fine-Tuning)

Use **Platt scaling** (logistic regression on predicted probabilities) to improve calibration.

In [ ]:
if TABPFN_AVAILABLE and base_model is not None:
    print("Applying calibration (Platt scaling)...")
    start_time = time.time()
    
    # Calibrate using cross-validation on training set
    calibrated_model = CalibratedClassifierCV(
        base_model,
        method='sigmoid',  # Platt scaling
        cv=5  # 5-fold CV
    )
    
    # Note: CalibratedClassifierCV will refit the base model on each fold
    calibrated_model.fit(X_train.values, y_train.values)
    
    calibration_time = time.time() - start_time
    print(f"✓ Calibration complete in {calibration_time:.2f} seconds")
    
    # Save calibrated model
    joblib.dump(calibrated_model, MODELS_DIR / "calibrated_model.joblib")
    print("✓ Saved calibrated model")
    
else:
    print("⚠ Demo mode: Cannot perform calibration without TabPFN")
    calibrated_model = None
    calibration_time = 0.0

## 5. Generate Fine-Tuned Predictions

In [ ]:
if TABPFN_AVAILABLE and calibrated_model is not None:
    # Predictions with calibrated model
    print("Generating calibrated predictions...")
    
    finetuned_train_proba = calibrated_model.predict_proba(X_train.values)[:, 1]
    finetuned_test_proba = calibrated_model.predict_proba(X_test.values)[:, 1]
    
    finetuned_train_pred = (finetuned_train_proba >= 0.5).astype(int)
    finetuned_test_pred = (finetuned_test_proba >= 0.5).astype(int)
    
    print("✓ Predictions generated")
    
else:
    # Use zero-shot predictions as placeholder
    print("⚠ Using zero-shot predictions (demo mode)")
    finetuned_test_proba = zeroshot_test['y_proba'].values
    finetuned_test_pred = zeroshot_test['y_pred'].values
    
    train_preds = zeroshot_preds[zeroshot_preds['split'] == 'train']
    finetuned_train_proba = train_preds['y_proba'].values
    finetuned_train_pred = train_preds['y_pred'].values

## 6. Performance Metrics

In [ ]:
# Compute metrics
finetuned_metrics = {
    'model': 'TabPFN (Fine-Tuned/Calibrated)',
    'mode': 'fine-tuned',
    'approach': approach,
    'train': {
        'auroc': float(roc_auc_score(y_train, finetuned_train_proba)),
        'auprc': float(average_precision_score(y_train, finetuned_train_proba)),
        'log_loss': float(log_loss(y_train, finetuned_train_proba)),
        'brier_score': float(brier_score_loss(y_train, finetuned_train_proba)),
        'accuracy': float(accuracy_score(y_train, finetuned_train_pred)),
        'precision': float(precision_score(y_train, finetuned_train_pred)),
        'recall': float(recall_score(y_train, finetuned_train_pred)),
        'f1': float(f1_score(y_train, finetuned_train_pred))
    },
    'test': {
        'auroc': float(roc_auc_score(y_test, finetuned_test_proba)),
        'auprc': float(average_precision_score(y_test, finetuned_test_proba)),
        'log_loss': float(log_loss(y_test, finetuned_test_proba)),
        'brier_score': float(brier_score_loss(y_test, finetuned_test_proba)),
        'accuracy': float(accuracy_score(y_test, finetuned_test_pred)),
        'precision': float(precision_score(y_test, finetuned_test_pred)),
        'recall': float(recall_score(y_test, finetuned_test_pred)),
        'f1': float(f1_score(y_test, finetuned_test_pred))
    },
    'efficiency': {
        'base_fit_time_seconds': float(fit_time),
        'calibration_time_seconds': float(calibration_time),
        'total_time_seconds': float(fit_time + calibration_time)
    }
}

print("TabPFN Fine-Tuned Performance (Test Set):")
print("="*60)
for metric, value in finetuned_metrics['test'].items():
    print(f"{metric:15s}: {value:.4f}")

# Save
with open(METRICS_DIR / "tabpfn_finetuned_metrics.json", 'w') as f:
    json.dump(finetuned_metrics, f, indent=2)
print("\n✓ Saved metrics")

## 7. Compare Zero-Shot vs Fine-Tuned

In [ ]:
# Create comparison table
comparison = pd.DataFrame([
    {
        'Model': 'Zero-Shot',
        'AUROC': zeroshot_metrics['test']['auroc'],
        'AUPRC': zeroshot_metrics['test']['auprc'],
        'Brier': zeroshot_metrics['test']['brier_score'],
        'Log Loss': zeroshot_metrics['test']['log_loss'],
        'Accuracy': zeroshot_metrics['test']['accuracy'],
        'F1': zeroshot_metrics['test']['f1']
    },
    {
        'Model': 'Fine-Tuned',
        'AUROC': finetuned_metrics['test']['auroc'],
        'AUPRC': finetuned_metrics['test']['auprc'],
        'Brier': finetuned_metrics['test']['brier_score'],
        'Log Loss': finetuned_metrics['test']['log_loss'],
        'Accuracy': finetuned_metrics['test']['accuracy'],
        'F1': finetuned_metrics['test']['f1']
    }
])

# Add difference row
diff_row = {'Model': 'Difference'}
for col in ['AUROC', 'AUPRC', 'Brier', 'Log Loss', 'Accuracy', 'F1']:
    diff_row[col] = comparison.loc[1, col] - comparison.loc[0, col]
comparison = pd.concat([comparison, pd.DataFrame([diff_row])], ignore_index=True)

print("\nZero-Shot vs Fine-Tuned Comparison:")
print("="*80)
print(comparison.to_string(index=False))

# Save
comparison.to_csv(TABLES_DIR / "tabpfn_zeroshot_vs_finetuned.csv", index=False)
print("\n✓ Saved comparison table")

## 8. Statistical Significance (DeLong Test)

In [ ]:
# DeLong test
delong_result = delong_test(
    y_test,
    finetuned_test_proba,
    zeroshot_test['y_proba'].values
)

print("DeLong Test: Fine-Tuned vs Zero-Shot")
print("="*60)
print(f"Fine-Tuned AUROC:  {delong_result['auroc1']:.4f}")
print(f"Zero-Shot AUROC:   {delong_result['auroc2']:.4f}")
print(f"Difference:        {delong_result['auroc1'] - delong_result['auroc2']:+.4f}")
print(f"Z-statistic:       {delong_result['z_statistic']:.4f}")
print(f"P-value:           {delong_result['p_value']:.4f}")
print(f"Significant (α=0.05): {'Yes' if delong_result['p_value'] < 0.05 else 'No'}")

# Effect size
nne_result = number_needed_to_evaluate(
    y_test,
    finetuned_test_pred,
    zeroshot_test['y_pred'].values
)

print(f"\nEffect Size (NNE): {nne_result['nne']:.1f}")
print(f"Interpretation: {nne_result['interpretation']}")

## 9. Calibration Improvement

In [ ]:
from sklearn.calibration import calibration_curve

# Compute calibration curves
prob_true_ft, prob_pred_ft = calibration_curve(
    y_test, finetuned_test_proba, n_bins=10, strategy='uniform'
)
prob_true_zs, prob_pred_zs = calibration_curve(
    y_test, zeroshot_test['y_proba'].values, n_bins=10, strategy='uniform'
)

# Plot
fig, ax = plt.subplots(figsize=(8, 8))

ax.plot(prob_pred_ft, prob_true_ft, marker='o', linewidth=2,
        label=f"Fine-Tuned (Brier={finetuned_metrics['test']['brier_score']:.4f})",
        color='red')
ax.plot(prob_pred_zs, prob_true_zs, marker='s', linewidth=2,
        label=f"Zero-Shot (Brier={zeroshot_metrics['test']['brier_score']:.4f})",
        color='blue', linestyle='--')
ax.plot([0, 1], [0, 1], 'k--', label='Perfect Calibration', linewidth=1)

ax.set_xlabel('Mean Predicted Probability', fontsize=12)
ax.set_ylabel('Fraction of Positives', fontsize=12)
ax.set_title('Calibration: Fine-Tuned vs Zero-Shot TabPFN', fontweight='bold', fontsize=13)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'tabpfn_finetuned_calibration_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved calibration comparison")

# Calibration improvement
brier_improvement = zeroshot_metrics['test']['brier_score'] - finetuned_metrics['test']['brier_score']
print(f"\nBrier score improvement: {brier_improvement:+.4f}")
print(f"  (Negative Brier is better; improvement = reduction)")

## 10. Performance Comparison Visualization

In [ ]:
# Bar plot comparing metrics
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics_to_plot = [
    ('AUROC', [zeroshot_metrics['test']['auroc'], finetuned_metrics['test']['auroc']]),
    ('AUPRC', [zeroshot_metrics['test']['auprc'], finetuned_metrics['test']['auprc']]),
    ('Brier Score', [zeroshot_metrics['test']['brier_score'], finetuned_metrics['test']['brier_score']]),
    ('F1 Score', [zeroshot_metrics['test']['f1'], finetuned_metrics['test']['f1']])
]

for ax, (metric_name, values) in zip(axes.flat, metrics_to_plot):
    x = np.arange(2)
    bars = ax.bar(x, values, color=['blue', 'red'], alpha=0.7)
    ax.set_xticks(x)
    ax.set_xticklabels(['Zero-Shot', 'Fine-Tuned'])
    ax.set_ylabel(metric_name, fontsize=11)
    ax.set_title(f'{metric_name} Comparison', fontweight='bold', fontsize=12)
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.4f}',
                ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'tabpfn_zeroshot_vs_finetuned_bars.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved comparison visualization")

## 11. Save Predictions

In [ ]:
# Save predictions
predictions = pd.DataFrame({
    'y_true': np.concatenate([y_train, y_test]),
    'y_pred': np.concatenate([finetuned_train_pred, finetuned_test_pred]),
    'y_proba': np.concatenate([finetuned_train_proba, finetuned_test_proba]),
    'split': ['train']*len(y_train) + ['test']*len(y_test)
})

predictions.to_parquet(PREDICTIONS_DIR / "tabpfn_finetuned_predictions.parquet", index=False)
print("✓ Saved predictions")

## Summary

**TabPFN Fine-Tuned Evaluation Complete:**
- ✓ Fine-tuning approach: {approach}
- ✓ Calibration applied (Platt scaling)
- ✓ Statistical comparison (DeLong test)
- ✓ Calibration improvement assessed
- ✓ Effect size calculated

**Key Findings:**
- Zero-shot AUROC: {:.4f}
- Fine-tuned AUROC: {:.4f}
- Difference: {:+.4f} (p={:.4f})
- Brier improvement: {:+.4f}
- Additional time cost: {:.1f} seconds

**Hypothesis Testing:**
- H1 (Performance): {} (AUROC difference {}significant)
- H2 (Calibration): {} (Brier score {})
- H3 (Fairness): To be evaluated in 05a_group_metrics.ipynb
- H4 (Efficiency): {:.1f}s additional cost

**Interpretation:**

**If fine-tuning improved performance:**
- Calibration likely improved (lower Brier score)
- Worth the computational cost for deployment
- But: check fairness implications
- And: risk of overfitting to COMPAS specifics

**If no significant improvement:**
- Zero-shot TabPFN already well-suited
- Simpler deployment (no calibration needed)
- Lower computational requirements
- More generalizable across datasets

**Recommendations:**
1. For research: Report both zero-shot and fine-tuned
2. For deployment: Use fine-tuned if calibration improves significantly
3. For fairness: Always evaluate group-specific impact (next notebooks)
4. For generalization: Test on other criminology datasets

**Next Steps:**
- 04c_tabpfn_vs_baselines.ipynb (Compare to all baselines)
- 04d_tabpfn_sensitivity.ipynb (Test robustness)
- 05a_group_metrics.ipynb (Fairness evaluation)